# Unitary Noise Analysis

This notebook analyzes how noisy the current unitary-compilation models are under quditkit global noise.

It focuses on:
- candidate circuits sampled from existing models
- clean compilation quality against the target unitary
- noisy degradation across a configurable `p` grid
- where the preferred candidate changes as noise increases


In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from notebooks.noise_awareness.unitary_noise_analysis_helper import (
    DEFAULT_P_GRID,
    build_model_specs,
    collect_candidate_tables,
    plot_noise_overview,
    save_analysis_bundle,
    summarize_candidate_tables,
)
from notebooks.noise_awareness.unitary_noise_common import DEFAULT_RESULTS_ROOT, maybe_dataframe


In [ ]:
DATASET_PATH = Path("./datasets/paper_qiskit/unitary_3q_dataset")
MODEL_DIRS = [
    Path("./models/trained/paper_unitary"),
    Path("./models/remote/qc_unitary_3qubit"),
]
TARGET_LIMIT = 16
SAMPLES_PER_TARGET = 16
P_GRID = list(DEFAULT_P_GRID)
GUIDANCE_SCALE = 4.0
AUTO_BATCH_SIZE = 64
CLEAN_INFidelity_THRESHOLD = 1e-6
NOISE_REALIZATIONS = 8
SEED = 0
DEVICE = "cpu"
OUTPUT_DIR = DEFAULT_RESULTS_ROOT / "unitary_noise_analysis_demo"

print("DATASET_PATH =", DATASET_PATH)
print("MODEL_DIRS =", MODEL_DIRS)
print("P_GRID =", P_GRID)
print("OUTPUT_DIR =", OUTPUT_DIR)


In [ ]:
model_specs = build_model_specs(MODEL_DIRS)
analysis = collect_candidate_tables(
    DATASET_PATH,
    model_specs=model_specs,
    target_limit=TARGET_LIMIT,
    samples_per_target=SAMPLES_PER_TARGET,
    p_grid=P_GRID,
    guidance_scale=GUIDANCE_SCALE,
    auto_batch_size=AUTO_BATCH_SIZE,
    clean_infidelity_threshold=CLEAN_INFidelity_THRESHOLD,
    noise_realizations=NOISE_REALIZATIONS,
    seed=SEED,
    device=DEVICE,
)
summary = summarize_candidate_tables(analysis["candidate_rows"], analysis["score_rows"])

display(maybe_dataframe(analysis["overview"]))


In [ ]:
print("Target overview")
display(maybe_dataframe(analysis["target_rows"][:20]))

print("Candidate summary by model")
display(maybe_dataframe(summary["by_model"]))

print("Noise summary by p")
display(maybe_dataframe(summary["by_noise_p"]))

print("Candidate rows")
display(maybe_dataframe(analysis["candidate_rows"][:50]))

print("Noise score rows")
display(maybe_dataframe(analysis["score_rows"][:50]))


In [ ]:
plot_noise_overview(analysis["candidate_rows"], analysis["score_rows"])


In [ ]:
saved_dir = save_analysis_bundle(analysis, output_dir=OUTPUT_DIR)
print("Saved analysis bundle to", saved_dir)


## Next Step

Once the candidate table looks reasonable, build a derived dataset from the terminal:

```bash
python notebooks/noise_awareness/build_unitary_noise_dataset.py \
  --source-dataset ./datasets/paper_qiskit/unitary_3q_dataset \
  --candidate-dir ./results/noise_aware_unitary/unitary_noise_analysis_demo \
  --output-dir ./datasets/noise_aware/unitary_3q_quditkit_noise_v1
```
